# 04 — TEP-shaped reactor-pressure transitions

Freeze a construction on one run and apply it unchanged to another. This
teaching fixture is not TEP source data and does not diagnose a fault.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import featuregraph as fg
from featuregraph.operators.states import rising_state

series = {
    "development": [2800, 2800, 2801, 2803, 2806, 2804, 2802,
                    2801, 2801, 2802, 2803, 2802, 2801, 2801],
    "heldout": [2800, 2800, 2801, 2804, 2807, 2805, 2802,
                2801, 2801, 2802, 2804, 2802, 2801, 2801],
}
observations = pd.concat([
    pd.DataFrame({"run": run, "sample": np.arange(len(x)),
                  "pressure_raw": x})
    for run, x in series.items()
], ignore_index=True)


The full TEP study uses a 50-sample rolling maximum,
rolling mean, and offline shift. This short lesson freezes a declared
three-sample centered mean for visibility.


In [ ]:
WINDOW = 3; EPSILON = .02
observations["pressure"] = observations.groupby("run")[
    "pressure_raw"
].transform(lambda x: x.rolling(WINDOW, center=True, min_periods=1).mean())
observations["rate"] = observations.groupby("run")["pressure"].diff().fillna(0)

rate = {"column": "rate"}; eps = {"parameter": "eps"}
contract = {
    "version": "state-contract-v1", "parameters": {"eps": EPSILON},
    "group_by": "run",
    "states": {
        "rising": {"op": "gt", "left": rate, "right": eps},
        "falling": {"op": "lt", "left": rate,
                    "right": {"op": "neg", "value": eps}},
        "inactive": {"op": "le", "left": {"op": "abs", "value": rate},
                     "right": eps},
    },
    "events": {"enter_rising": {"type": "enter_state", "state": "rising"},
               "exit_rising": {"type": "exit_state", "state": "rising"}},
}
compiled = fg.compile_states(observations, contract)


In [ ]:
objects = compiled.observations.groupby(
    ["run", "state_occurrence_id", "state"], sort=False
).agg(
    start_sample=("sample", "min"), end_sample=("sample", "max"),
    duration=("sample", "size"), start_pressure=("pressure", "first"),
    end_pressure=("pressure", "last"),
).reset_index()
objects["net_change"] = objects["end_pressure"] - objects["start_pressure"]
objects


In [ ]:
development = observations.query("run == 'development'").copy()
rising = fg.transition.Transition(
    development, "pressure", "rising", rising_state, eps=EPSILON
)

summary = compiled.observations.groupby("run").agg(
    maximum_pressure=("pressure", "max"),
    rising_entries=("enter_rising", "sum"),
)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for run, part in compiled.observations.groupby("run", sort=False):
    ax.plot(part["sample"], part["pressure"], marker="o", label=run)
ax.legend(); ax.grid(alpha=.2); plt.show()


In [ ]:
assert compiled.validation_report["passed"].all()
assert compiled.observations.groupby("run")[
    "state_occurrence_id"
].min().eq(0).all()
assert summary.loc["heldout", "maximum_pressure"] > summary.loc[
    "development", "maximum_pressure"
]
assert set(objects["state"]) == {"rising", "falling", "inactive"}


The larger held-out peak is descriptive, not a fault
identifier. The maintained TEP study transfers across held-out Fault 2 runs,
normal windows, and contrasting faults, showing that pressure is not specific.
